In [0]:
# Imports
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality, join_on
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, get_metrics, available_metrics, Rollup, Cube
import pyspark.sql.functions as f
from pyspark.sql.types import *
import re
import os
import sys
import time
import upc_input
import datetime as dt
from seg import profile
from pyspark.sql.window import Window
from poirot import SparkManager
from pyspark.sql.types import DoubleType

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
kpi = KPI(use_sample_mart = False, apply_privacy_filters = True)
acds = ACDS(use_sample_mart = False, apply_privacy_filters = True)

#### Single Channel XCM View of KPF Dashboard
- Contains only closed loop XCMs by Channel View. (Channels per campaign distinguished by KPM_project_ID)
- Non-Closed Loop Campaigns not included.
- Contains engagement metrics (Redemptions, Downloads, ect) and uplift metrics by exposure type.
- TO-DO: Pull MMCI table to get campaign cost by channel.

<img src="./Screenshot 2026-01-29 120500.png" alt="Screenshot 2026-01-29 120500.png" title="Screenshot 2026-01-29 120500.png" width="1000"/>

#### Mhtv + MMCI Uplift Metrics

##### 2025 and before (XCM) Single Channel View

In [0]:
# Code to generate closed loop campaign tab

# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.
media_history_revamped = (
    spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
    .filter(f.col("campaign_id").rlike("^[0-9]+$"))
    .filter(f.col("camp_start_date").isNotNull())
).cache()
media_history_revamped.count() 

mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.withColumn(
  "MANUFACTURER", f.when(f.col("KPM_PROJECT_ID") == 93220, "Kroger Personal Finance").otherwise(f.col("MANUFACTURER"))
)

# Rename and Join Media History and Campaign Availability Tables
mhtv_mmci = media_history_revamped.join(
    mmci.select('kpm_duplicated_id', 'channel', 'KROGER_START_WEEK', 'Fiscal_Start_Year', 'Fiscal_Quarter', 'target_id', "SIGNED_OFF")
        .withColumnRenamed('Fiscal_Start_Year','year')
        .withColumnRenamed('Fiscal_Quarter','quarter')
        .withColumnRenamed('kpm_duplicated_id','campaign_id')
        .distinct(),
    ['campaign_id'],
    'inner'
)

# Hard Coding to remove incorrect product groups and discrepancies (from legacy code)
mhtv_mmci = (
    mhtv_mmci
    .filter(~f.col('product_group').like('%800000016326%'))
    .filter(~((f.col('campaign_id') == 49863) & (~f.lower(f.col('product_group')).like('%category%'))))
    .filter(~((f.col('campaign_id') != 49863) & (f.lower(f.col('product_group')).like('%category%'))))
    .filter(~f.col('product_group').like('%OL variable load%'))
    .filter(~((f.col('campaign_id') == 37539) & (f.col('adjusted_top_performer') == 'Not-Top-Performer_KRO')))
    .filter(~((f.col('campaign_id') == 89949) & (f.col('quarter') == 'Q1')))
)

mhtv_mmci.display()

In [0]:
mhtv_mmci_filtered = mhtv_mmci.filter(
    (f.col('KROGER_START_WEEK') >= '20220101') & (f.col('KROGER_START_WEEK') < '20260201') &
    (f.col("manufacturer").isin("Kroger Wallet", "Kroger Personal Finance")) &
    (f.col("SIGNED_OFF") == "Y") & 
    (f.col("modality") == "All Modalities") &
    (f.col("rom") == "Kroger Only")
)

# 1. Updated Top Performer Logic: For NON XCMs, If a campaign has an offer attached, then 800's product groups will be top performers.
# If a campaign has NO offer attached, then the flag "Adjusted_Top_Performer" will be used to obtain top performers
# If a campaign has 800s product group attached but ALSO 4x (Regional), then use adjusted top performer flag (for example SSE Gift).
campaign_window = Window.partitionBy("campaign_id")
mhtv_mmci_filtered_non_xcm = mhtv_mmci_filtered.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception")

# 2. Updated Top Performer Logic: For XCMs, Adjusted Top Performers == "TOP-PERFORMER_KRO" will be Top Performers.
mhtv_mmci_filtered_xcm = mhtv_mmci_filtered.filter(f.col("adjusted_top_performer") == "Top-Performer_KRO")

# Could have done union after upstream calculations. Did it this way to QC intermediate outputs as well.
mhtv_mmci_filtered_closed_loop_pre_2026 = mhtv_mmci_filtered_non_xcm.unionByName(mhtv_mmci_filtered_xcm )

mhtv_mmci_filtered_closed_loop_pre_2026.display()

##### 2026 and onwards Single Channel (XCM deprecated)

In [0]:
# Code to generate closed loop campaign tab

# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.
media_history_revamped = (
    spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
    .filter(f.col("campaign_id").rlike("^[0-9]+$"))
    .filter(f.col("kpm_duplicated_id").rlike("^[0-9]+$"))
    .filter(f.col("camp_start_date").isNotNull())
).cache()
media_history_revamped.count() 

mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.withColumn(
  "MANUFACTURER", f.when(f.col("KPM_PROJECT_ID") == 93220, "Kroger Personal Finance").otherwise(f.col("MANUFACTURER"))
)

# Rename and Join Media History and Campaign Availability Tables: Join on duplicated == mmci.project id to capture all single channels
mhtv_mmci = media_history_revamped.join(
    mmci.select('KPM_PROJECT_ID', 'channel', 'KROGER_START_WEEK', 'Fiscal_Start_Year', 'Fiscal_Quarter', 'target_id', "SIGNED_OFF")
        .withColumnRenamed('Fiscal_Start_Year','year')
        .withColumnRenamed('Fiscal_Quarter','quarter')
        .distinct(),
    media_history_revamped.kpm_duplicated_id == mmci.KPM_PROJECT_ID,
    'inner'
).drop('KPM_PROJECT_ID')

# Hard Coding to remove incorrect product groups and discrepancies (from legacy code)
mhtv_mmci = (
    mhtv_mmci
    .filter(~f.col('product_group').like('%800000016326%'))
    .filter(~((f.col('campaign_id') == 49863) & (~f.lower(f.col('product_group')).like('%category%'))))
    .filter(~((f.col('campaign_id') != 49863) & (f.lower(f.col('product_group')).like('%category%'))))
    .filter(~f.col('product_group').like('%OL variable load%'))
    .filter(~((f.col('campaign_id') == 37539) & (f.col('adjusted_top_performer') == 'Not-Top-Performer_KRO')))
    .filter(~((f.col('campaign_id') == 89949) & (f.col('quarter') == 'Q1')))
)

mhtv_mmci_filtered_2026 = mhtv_mmci.filter(
    (f.col('KROGER_START_WEEK') >= '20260201') &
    (f.col("manufacturer").isin("Kroger Wallet", "Kroger Personal Finance")) &
    (f.col("SIGNED_OFF") == "Y") & 
    (f.col("modality") == "All Modalities") &
    (f.col("rom") == "Kroger Only")
)

mhtv_mmci_filtered_2026.display()

In [0]:
# 1. Updated Top Performer Logic: For NON XCMs, If a campaign has an offer attached, then 800's product groups will be top performers.
# If a campaign has NO offer attached, then the flag "Adjusted_Top_Performer" will be used to obtain top performers
# If a campaign has 800s product group attached but ALSO 4x (Regional), then use adjusted top performer flag (for example SSE Gift).
campaign_window = Window.partitionBy("campaign_id")
mhtv_mmci_filtered_non_xcm_2026 = mhtv_mmci_filtered_2026.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception")

# 2. Updated Top Performer Logic: For XCMs, Adjusted Top Performers == "TOP-PERFORMER_KRO" will be Top Performers.
mhtv_mmci_filtered_xcm_2026 = mhtv_mmci_filtered_2026.filter(f.col("adjusted_top_performer") == "Top-Performer_KRO")

# Could have done union after upstream calculations. Did it this way to QC intermediate outputs as well.
mhtv_mmci_filtered_closed_loop_2026 = mhtv_mmci_filtered_non_xcm_2026.unionByName(mhtv_mmci_filtered_xcm_2026)

mhtv_mmci_filtered_closed_loop_2026.display()

In [0]:
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop_2026.unionByName(mhtv_mmci_filtered_closed_loop_pre_2026)
mhtv_mmci_filtered_closed_loop.filter(f.col("year") == 2026).display()

##### KPF specific filters + aggregation

In [0]:
# Business Line based on Project Name. We need this column, as for Single Channel View we need to filter to only Closed Loop XCMs (OL, Gift, Account Funding, Lottery)
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.withColumn(
    "business_line",
    f.when(
         f.upper(f.col("project_name")).contains("LOTT"), "Lottery"
    ).when(
        f.upper(f.col("project_name")).contains("OPEN LOOP") | 
        f.upper(f.col("project_name")).contains(" OL "), "Open Loop"
    ).when(
        f.upper(f.col("project_name")).contains("LOCAL") |
        f.upper(f.col("project_name")).contains("TDC KPF") |
        f.upper(f.col("project_name")).contains("TDC SFID") |
        f.upper(f.col("project_name")).contains("TDC SFPRJ") |
        f.upper(f.col("project_name")).contains("MCP") |
        f.upper(f.col("project_name")).contains("BULK") |
        f.upper(f.col("project_name")).contains(" 3P GIFT ") |
        f.upper(f.col("project_name")).contains("GIFT"), "Gift"
    ).when(
        f.upper(f.col("project_name")).contains("MONEY SERVICES") |
        f.upper(f.col("project_name")).contains(" MS "), "Money Services"
    ).when(
        f.upper(f.col("project_name")).contains(" PAY "), "Kroger Pay"
    ).when(
        f.upper(f.col("project_name")).contains("KROGER WALLET"), "Kroger Wallet"
    ).when(
        f.upper(f.col("project_name")).contains(" ACH "), "Debit"
    ).when(
        f.upper(f.col("project_name")).contains(" CICO "), "Account Funding"
    ).when(
        f.upper(f.col("project_name")).contains(" SBE "), "SBE"
    ).when(
        f.upper(f.col("project_name")).contains("CREDIT"), "Credit"
    ).when(
        f.upper(f.col("project_name")).contains(" REM "), "Gift"
    ).otherwise("")
)


mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.filter(
    (f.col("campaign_type") == "XCM") | 
    (f.col("project_name").contains("XCM SSE")) |
    ((f.col("project_name").contains("XCM") & (f.col("year") == 2026)))
)

# Filter to ONLY Closed Loop XCMs
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.filter(f.col("business_line").isin("Open Loop", 
"Gift", "Account Funding", "Lottery"))
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.select("campaign_id", "target_id", "year", "quarter", "camp_start_date", "camp_end_date", "job_id", "project_name", "channel", "campaign_type", "business_line", "camp_cost", "test_hh_count",  "sales_test_total", "sales_uplift_total", "sales_uplift_pct", "hhpen_test_total", "hhpen_uplift_total", "hhpen_uplift_pct", "visits_test_total", "visits_uplift_total", "visits_uplift_pct", "units_test_total", "units_uplift_total", "units_uplift_pct").dropDuplicates()

mhtv_mmci_filtered_closed_loop.display()


In [0]:
# Hard Coding uplift metrics for a small few campaigns where metadata is not reflecting wrap report: 138143
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.0108)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.0078)).otherwise(f.col("visits_uplift_pct"))
)

# Hard Coding a small few campaigns where metadata is not reflecting wrap report: 147403
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0327)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0110)).otherwise(f.col("visits_uplift_pct"))
).withColumn(
    "units_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0119)).otherwise(f.col("visits_uplift_pct"))
)

# Hard Coding a small few campaigns where metadata is not reflecting wrap report: 82541
mhtv_mmci_filtered_closed_loop = mhtv_mmci_filtered_closed_loop.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0374)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0168)).otherwise(f.col("visits_uplift_pct"))
).withColumn(
    "units_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0348)).otherwise(f.col("visits_uplift_pct"))
)

mhtv_mmci_filtered_closed_loop.display()

#### Media Metrics: Engagement Metrics

In [0]:
# pull media metrics for engagement metrics
# Joining on mhtv campaign id, but also on mhtv job id to get only the latest job id
media_metrics = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure').filter( (f.col('segment') == 'ALL'))
media_metrics = media_metrics.withColumnRenamed("kpm_project_id", "campaign_id")

mhtv_mmci_mmet_xcm = mhtv_mmci_filtered_closed_loop.join(
  media_metrics.select("campaign_id", "job_id", "metric", "value", "rom", "modality", "metrics_type"),
  on=[mhtv_mmci_filtered_closed_loop.campaign_id == media_metrics.campaign_id,
      mhtv_mmci_filtered_closed_loop.job_id == media_metrics.job_id],
  how='inner'
).drop(media_metrics.job_id, media_metrics.campaign_id, media_metrics.campaign_type)

mhtv_mmci_mmet_xcm.display()



In [0]:
mhtv_mmci_mmet_xcm_filtered = mhtv_mmci_mmet_xcm.filter(
    (f.col('camp_start_date') >= '2022-01-01') &
    (f.col("modality") == "All Modalities") &
    (f.col("rom") == "Kroger Only")
).dropDuplicates()

mhtv_mmci_mmet_xcm_filtered.display()


In [0]:
# Pivoting Media Metrics Data to get metrics for each XCM tactic type (including NULL for non-applicable metrics)
mhtv_mmci_mmet_xcm_pivot = (
    mhtv_mmci_mmet_xcm_filtered
    .filter(f.col("metric").isNotNull())
    .groupBy(
        "campaign_id", "target_id", "campaign_type", "channel", "year", "quarter", "business_line",
        "project_name", "job_id", "camp_cost",
        "metrics_type", "camp_start_date", "camp_end_date", "test_hh_count",
        "sales_test_total", "sales_uplift_total", "sales_uplift_pct",
        "hhpen_test_total", "hhpen_uplift_total", "hhpen_uplift_pct",
        "visits_test_total", "visits_uplift_total", "visits_uplift_pct",
        "units_test_total", "units_uplift_total", "units_uplift_pct"
    )
    .pivot("metric")
    .agg(f.first("value"))
).dropDuplicates()

display(mhtv_mmci_mmet_xcm_pivot)

In [0]:
# Keep only certain engagement metrics, and include it with uplift metrics and campaign information
engagement_metrics = [
    "unique_clicked", "clickthrough_rate", 
    "open_rate", "total_downloads", 
    "total_impressions", "viewability_percent"
]

filtered_mhtv_mmet_top_performer_pivot = mhtv_mmci_mmet_xcm_pivot.select(
   "campaign_id", "target_id", "campaign_type", "channel", "year", "quarter", "business_line", "project_name", "job_id", "camp_cost",
        "metrics_type", "camp_start_date", "camp_end_date", "test_hh_count",
        "sales_test_total", "sales_uplift_total", "sales_uplift_pct",
        "hhpen_test_total", "hhpen_uplift_total", "hhpen_uplift_pct",
        "visits_test_total", "visits_uplift_total", "visits_uplift_pct",
        "units_test_total", "units_uplift_total", "units_uplift_pct", *engagement_metrics
).orderBy("campaign_id", "channel")

display(filtered_mhtv_mmet_top_performer_pivot)

In [0]:
# Separate XCMs into distinct tactics based on metrics_type column
single_channel_view_df = filtered_mhtv_mmet_top_performer_pivot.filter(
    ((f.col("metrics_type") == "email_metrics_EMOD") & (f.col("channel") == "Email Module")) |
    ((f.col("metrics_type") == "email_metrics_SSE") & (f.col("channel") == "Single Subject Email")) |
    ((f.col("metrics_type") == "offsite_DISPLAY_AD") & (f.col("channel") == "Display Ad")) |
    ((f.col("metrics_type") == "push_metrics") & (f.col("channel") == "Push Notifications")) |
    ((f.col("metrics_type") == "offsite_PAND") & (f.col("channel") == "Pandora")) |
    ((f.col("metrics_type") == "offsite_PRV") & (f.col("channel") == "Pre-Roll Video")) | 
    ((f.col("year") == 2026) & ((f.col("metrics_type") == "push_metrics") &  (f.col("campaign_type") == "PUSH"))) |
    ((f.col("year") == 2026) & ((f.col("metrics_type") == "email_metrics_SSE") &  (f.col("campaign_type") == "SSE"))) |
    ((f.col("year") == 2026) & ((f.col("metrics_type") == "offsite_PINT") &  (f.col("campaign_type") == "PINT"))) |
    ((f.col("year") == 2026) & ((f.col("metrics_type") == "offsite_DISPLAY_AD") &  (f.col("campaign_type") == "DISPLAY_AD"))) |
    ((f.col("year") == 2026) & ((f.col("metrics_type") == "offsite_PRV") &  (f.col("campaign_type") == "PRV"))) 
)

single_channel_view_df.display()

In [0]:
# mmci campaign cost is null for single channels in 2026, but they are already included in mhtv
single_channel_view_df_2026 = single_channel_view_df.filter(f.col("year") == 2026)
single_channel_view_df_pre_2026 = single_channel_view_df.filter(f.col("year") < 2026)

In [0]:
# Add true camp cost per xcm channel
campaign_info_cost_per_xcm_channel = mmci.select("KPM_DUPLICATED_ID", "CHANNEL", "TOT_COST")

single_channel_view_df_pre_2026 = single_channel_view_df_pre_2026.drop("camp_cost").join(
    campaign_info_cost_per_xcm_channel
        .withColumnRenamed("KPM_DUPLICATED_ID", "campaign_id")
        .withColumnRenamed("CHANNEL", "channel")
        .withColumnRenamed("TOT_COST", "camp_cost"),
    on=["campaign_id", "channel"],
    how="left"
).dropDuplicates()

single_channel_view_df_pre_2026.display()

In [0]:
single_channel_view_df = single_channel_view_df_pre_2026.unionByName(single_channel_view_df_2026)
display(single_channel_view_df)

In [0]:
# Update channel for 2026 REM SSE PUSH and Offsites
single_channel_view_df = single_channel_view_df.withColumn(
    "channel",
    f.when((f.col("year") == 2026) & (f.col("campaign_type") == "PUSH"), "Push Notifications")
     .when((f.col("year") == 2026) & (f.col("campaign_type") == "SSE"), "Single Subject Email")
     .when((f.col("year") == 2026) & (f.col("campaign_type") == "PRV"), "Pre-Roll Video")
     .when((f.col("year") == 2026) & (f.col("campaign_type") == "DISPLAY_AD"), "Display Ad")
     .when((f.col("year") == 2026) & (f.col("campaign_type") == "PINT"), "Pinterest")
     .otherwise(f.col("channel"))
)

display(single_channel_view_df)

In [0]:
# User defined melt to unpivot our data (not built into Pyspark)
def melt(df, id_vars, value_vars, var_name = "variable", value_name = "value"):
    n = len(value_vars)
    expr = ", ".join([f"'{c}', {c}" for c in value_vars])
    return df.selectExpr(
        *id_vars,
        f"stack({n}, {expr}) as ({var_name}, {value_name})"
    )

In [0]:
# melt engagment metrics for Jenny to use
campaign_info = ["campaign_id", "target_id", "project_name", "job_id", "channel", "campaign_type", "year", "quarter", "business_line", "camp_start_date", "camp_end_date"]
# Drop downloads and redemptions, as per discussions w/ Shumaila and Lauren
uplift_engagement_metrics = ["test_hh_count", "camp_cost", "sales_test_total", "sales_uplift_total", "sales_uplift_pct",
        "hhpen_test_total", "hhpen_uplift_total", "hhpen_uplift_pct",
        "visits_test_total", "visits_uplift_total", "visits_uplift_pct",
        "units_test_total", "units_uplift_total", "units_uplift_pct",
        "unique_clicked", "clickthrough_rate", 
        "open_rate", 
        "total_impressions", "viewability_percent"]

single_channel_view_df_casted = single_channel_view_df
for col_name in uplift_engagement_metrics:
    single_channel_view_df_casted = single_channel_view_df_casted.withColumn(col_name, f.col(col_name).cast(DoubleType()))

single_channel_view_final = melt(single_channel_view_df_casted, id_vars=campaign_info,
    value_vars=uplift_engagement_metrics, var_name="metric", value_name="value"
).dropDuplicates()

single_channel_view_final = single_channel_view_final.filter(
    f.col('camp_start_date') >= '2023-01-01')

single_channel_view_final.display()

In [0]:
single_channel_view_final.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/single_channel_view.csv')
                                                                                                                            